# SatDiff — Colab training

## One-time setup

**Runtime → Change runtime type → T4 GPU.** That's it — the repo is public, so there is no token and no secret to configure.

## Every session

Run all cells top to bottom. Cell 5 resumes from the last checkpoint on its own — a disconnect costs you one epoch, never the run.

Checkpoints, sample grids, and `experiments.csv` all live in Drive. Only the dataset lives in `/content`, and it re-downloads in about 2 minutes.

## Before the real run

Do a one-epoch dry run first, with `configs/smoke.yaml` and throwaway output dirs, so a crash costs 5 minutes instead of surfacing 2 hours in:

```
!python -m satdiff.train --config configs/smoke.yaml \
    --checkpoint-dir $CKPT_DIR/smoke --results-dir $RESULTS_DIR/smoke
!python -m satdiff.eval --config configs/smoke.yaml --split val \
    --checkpoint-dir $CKPT_DIR/smoke --results-dir $RESULTS_DIR/smoke
```

In [ ]:
# 1. GPU check — if this says False, stop and fix the runtime type.
import torch
print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# 2. Drive. Everything you would be sad to lose gets written here.
import os
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR    = '/content/drive/MyDrive/satdiff/checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/satdiff/results'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("checkpoints ->", CKPT_DIR)
print("grids + csv ->", RESULTS_DIR)

In [ ]:
# 3. Code + deps. The repo is public, so this clones over plain HTTPS — no
# token, no Colab secret, nothing to set up.
import os, sys

CLONE_URL = 'https://github.com/DrKingSchultz69/satdiff-v2.git'

%cd /content
if os.path.isdir('/content/satdiff-v2/.git'):
    !cd /content/satdiff-v2 && git pull -q
else:
    !git clone -q $CLONE_URL
%cd /content/satdiff-v2

!pip install -q -r requirements.txt

os.environ['PYTHONPATH'] = '/content/satdiff-v2/src'
sys.path.insert(0, '/content/satdiff-v2/src')
print('ready')

In [ ]:
# 4. Data — 94 MB, ~2 min. Stays in /content on purpose: writing 27,000 small
# files to Drive is far slower than just re-downloading them each session.
!python scripts/download_data.py
!python scripts/make_splits.py

In [ ]:
# 5. Train. Re-running this cell after a disconnect is safe — --resume picks
# up from the last completed epoch.
#
# Every epoch prints its duration and the estimated time remaining.
# 100 epochs is roughly 7h on a T4, so expect this to span 2-3 sessions.
!python -m satdiff.train --config configs/v1.yaml --checkpoint-dir $CKPT_DIR --results-dir $RESULTS_DIR --resume

In [ ]:
# 6. The eye test. One row per class, same 4 seeds every epoch.
# Four identical images in a row = mode collapse, no matter what KID says.
import glob
from IPython.display import Image, display

grids = sorted(glob.glob(f'{RESULTS_DIR}/grids/*.png'))
if grids:
    print(grids[-1])
    display(Image(grids[-1]))
else:
    print('none yet — the first grid is written at epoch 5')

In [ ]:
# 7. Eval: KID + CAS. Run this once you are past ~epoch 20.
# CAS below 40% means conditioning is not learning — stop and fix the
# architecture rather than burning more GPU hours. See docs/eval-plan.md.
!python -m satdiff.eval --config configs/v1.yaml --split val --checkpoint-dir $CKPT_DIR --results-dir $RESULTS_DIR

In [ ]:
# 8. Every eval run so far, newest last.
import os
import pandas as pd

csv = f'{RESULTS_DIR}/experiments.csv'
display(pd.read_csv(csv)) if os.path.exists(csv) else print('no eval runs yet — run cell 7')